# ETL Pipeline for Retail Store Sales

## Extract, Transform, Load process for retail_store_sales.csv dataset

## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: EXTRACT - Load the CSV Data

In [2]:
# Extract: Read the CSV file
df = pd.read_csv('retail_store_sales.csv')

print(f"Data extracted successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

Data extracted successfully!
Shape: (12575, 11)

First few rows:


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [3]:
# Check data info and missing values
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nMissing Values:")
print(df.isnull().sum())
print("\n" + "="*50)
print("\nData Types:")
print(df.dtypes)

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.1+ MB
None


Missing Values:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method     

## Step 3: TRANSFORM - Clean and Process the Data

In [4]:
# Create a copy for transformation
df_transformed = df.copy()

print("Starting data transformation...")
print(f"Initial shape: {df_transformed.shape}")

Starting data transformation...
Initial shape: (12575, 11)


In [5]:
# 1. Handle missing values in numeric columns
# Replace empty strings with NaN for proper handling
df_transformed.replace('', np.nan, inplace=True)

# Check missing values after replacement
print("Missing values after converting empty strings:")
print(df_transformed.isnull().sum())

Missing values after converting empty strings:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [6]:
# 2. Convert data types
# Convert numeric columns
df_transformed['Price Per Unit'] = pd.to_numeric(df_transformed['Price Per Unit'], errors='coerce')
df_transformed['Quantity'] = pd.to_numeric(df_transformed['Quantity'], errors='coerce')
df_transformed['Total Spent'] = pd.to_numeric(df_transformed['Total Spent'], errors='coerce')

# Convert Transaction Date to datetime
df_transformed['Transaction Date'] = pd.to_datetime(df_transformed['Transaction Date'], errors='coerce')

print("Data types converted successfully!")
print(df_transformed.dtypes)

Data types converted successfully!
Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied            object
dtype: object


In [7]:
# 3. Handle missing values in numeric columns
# Fill missing Price Per Unit with median grouped by Category
df_transformed['Price Per Unit'] = df_transformed.groupby('Category')['Price Per Unit'].transform(
    lambda x: x.fillna(x.median())
)

# Fill missing Quantity with median
df_transformed['Quantity'].fillna(df_transformed['Quantity'].median(), inplace=True)

# Recalculate Total Spent where missing (Price Per Unit * Quantity)
df_transformed['Total Spent'] = df_transformed.apply(
    lambda row: row['Price Per Unit'] * row['Quantity'] 
    if pd.isna(row['Total Spent']) else row['Total Spent'], 
    axis=1
)

print("Numeric missing values handled!")
print("\nRemaining missing values in numeric columns:")
print(df_transformed[['Price Per Unit', 'Quantity', 'Total Spent']].isnull().sum())

C:\Users\mausa\AppData\Local\Temp\ipykernel_18804\3649358136.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_transformed['Quantity'].fillna(df_transformed['Quantity'].median(), inplace=True)


Numeric missing values handled!

Remaining missing values in numeric columns:
Price Per Unit      0
Quantity          604
Total Spent       604
dtype: int64


In [8]:
# 4. Handle missing values in categorical columns
# Fill missing Item with 'Unknown'
df_transformed['Item'].fillna('Unknown', inplace=True)

# Handle Discount Applied column (convert to boolean)
# Map True/False strings to boolean, NaN to False
df_transformed['Discount Applied'] = df_transformed['Discount Applied'].map({
    'True': True, 
    'False': False, 
    True: True, 
    False: False
})
df_transformed['Discount Applied'].fillna(False, inplace=True)

print("Categorical missing values handled!")
print("\nMissing values after transformation:")
print(df_transformed.isnull().sum())

Categorical missing values handled!

Missing values after transformation:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


C:\Users\mausa\AppData\Local\Temp\ipykernel_18804\774224061.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_transformed['Item'].fillna('Unknown', inplace=True)
C:\Users\mausa\AppData\Local\Temp\ipykernel_18804\774224061.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assig

In [9]:
# 5. Remove duplicates if any
initial_rows = len(df_transformed)
df_transformed.drop_duplicates(inplace=True)
removed_duplicates = initial_rows - len(df_transformed)

print(f"Removed {removed_duplicates} duplicate rows")
print(f"Final shape: {df_transformed.shape}")

Removed 0 duplicate rows
Final shape: (12575, 11)


In [10]:
# 6. Verify transformed data
print("Transformation Summary:")
print("="*60)
print(f"Total Records: {len(df_transformed)}")
print(f"\nData Types:")
print(df_transformed.dtypes)
print(f"\nMissing Values:")
print(df_transformed.isnull().sum())
print("\n" + "="*60)
print("\nSample of transformed data:")
df_transformed.head(10)

Transformation Summary:
Total Records: 12575

Data Types:
Transaction ID                 str
Customer ID                    str
Category                       str
Item                           str
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
Discount Applied            object
dtype: object

Missing Values:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


Sample of transformed data:


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
5,TXN_7482416,CUST_09,Patisserie,NaN,23.0,10.0,200.0,Credit Card,Online,2023-11-30,NaN
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True
8,TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,27.5,Credit Card,In-store,2023-04-26,False
9,TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,109.5,Cash,Online,2024-03-14,False


In [11]:
# Fix remaining missing values (due to pandas Copy-on-Write behavior)
# Handle Item - fill with 'Unknown'
df_transformed = df_transformed.assign(
    Item=df_transformed['Item'].fillna('Unknown')
)

# Handle Quantity - fill with median
df_transformed = df_transformed.assign(
    Quantity=df_transformed['Quantity'].fillna(df_transformed['Quantity'].median())
)

# Handle Total Spent - recalculate from Price Per Unit * Quantity where missing
df_transformed = df_transformed.assign(
    Total_Spent=df_transformed.apply(
        lambda row: row['Price Per Unit'] * row['Quantity'] 
        if pd.isna(row['Total Spent']) else row['Total Spent'], 
        axis=1
    )
)
df_transformed = df_transformed.drop(columns=['Total Spent']).rename(columns={'Total_Spent': 'Total Spent'})

# Handle Discount Applied - fill with False
df_transformed = df_transformed.assign(
    Discount_Applied=df_transformed['Discount Applied'].fillna(False)
)
df_transformed = df_transformed.drop(columns=['Discount Applied']).rename(columns={'Discount_Applied': 'Discount Applied'})

print("All missing values fixed!")
print("\nFinal missing values check:")
print(df_transformed.isnull().sum())

All missing values fixed!

Final missing values check:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Payment Method      0
Location            0
Transaction Date    0
Total Spent         0
Discount Applied    0
dtype: int64


## Step 4: LOAD - Save Data to SQLite3 Database

In [12]:
# Create SQLite database connection
db_name = 'retail_sales.db'
table_name = 'sales_data'

# Connect to SQLite database (creates if doesn't exist)
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

print(f"Connected to database: {db_name}")

Connected to database: retail_sales.db


In [13]:
# Load data into SQLite database
# if_exists='replace' will drop the table if it exists and create a new one
df_transformed.to_sql(table_name, conn, if_exists='replace', index=False)

print(f"Data loaded successfully into table: {table_name}")
print(f"Total records loaded: {len(df_transformed)}")

Data loaded successfully into table: sales_data
Total records loaded: 12575


In [14]:
# Verify the data in SQLite database
# Get table information
cursor.execute(f"PRAGMA table_info({table_name})")
columns = cursor.fetchall()

print("Table Schema:")
print("="*60)
for col in columns:
    print(f"Column: {col[1]}, Type: {col[2]}")
print("="*60)

Table Schema:
Column: Transaction ID, Type: TEXT
Column: Customer ID, Type: TEXT
Column: Category, Type: TEXT
Column: Item, Type: TEXT
Column: Price Per Unit, Type: REAL
Column: Quantity, Type: REAL
Column: Payment Method, Type: TEXT
Column: Location, Type: TEXT
Column: Transaction Date, Type: TIMESTAMP
Column: Total Spent, Type: REAL
Column: Discount Applied, Type: INTEGER


In [15]:
# Query and display sample data from database
query = f"SELECT * FROM {table_name} LIMIT 10"
df_from_db = pd.read_sql_query(query, conn)

print("Sample data from SQLite database:")
df_from_db

Sample data from SQLite database:


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Payment Method,Location,Transaction Date,Total Spent,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,Digital Wallet,Online,2024-04-08 00:00:00,185.0,1
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,Digital Wallet,Online,2023-07-23 00:00:00,261.0,1
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,Credit Card,Online,2022-10-05 00:00:00,43.0,0
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,Credit Card,Online,2022-05-07 00:00:00,247.5,0
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,Digital Wallet,Online,2022-10-02 00:00:00,87.5,0
5,TXN_7482416,CUST_09,Patisserie,Unknown,23.0,10.0,Credit Card,Online,2023-11-30 00:00:00,200.0,0
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,Credit Card,In-store,2023-06-10 00:00:00,40.0,1
7,TXN_1372952,CUST_21,Furniture,Unknown,33.5,6.0,Digital Wallet,In-store,2024-04-02 00:00:00,201.0,1
8,TXN_9728486,CUST_23,Furniture,Item_16_FUR,27.5,1.0,Credit Card,In-store,2023-04-26 00:00:00,27.5,0
9,TXN_2722661,CUST_25,Butchers,Item_22_BUT,36.5,3.0,Cash,Online,2024-03-14 00:00:00,109.5,0


In [16]:
# Get row count from database
cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
count = cursor.fetchone()[0]

print(f"Total records in database: {count}")

# Get some statistics
stats_query = f"""
SELECT 
    COUNT(*) as total_transactions,
    SUM("Total Spent") as total_revenue,
    AVG("Total Spent") as avg_transaction_value,
    MIN("Transaction Date") as earliest_date,
    MAX("Transaction Date") as latest_date
FROM {table_name}
"""

stats = pd.read_sql_query(stats_query, conn)
print("\nDatabase Statistics:")
print("="*60)
print(stats.to_string(index=False))

Total records in database: 12575

Database Statistics:
 total_transactions  total_revenue  avg_transaction_value       earliest_date         latest_date
              12575      1637367.0             130.208111 2022-01-01 00:00:00 2025-01-18 00:00:00


In [17]:
# Close database connection
conn.close()
print("Database connection closed.")
print("\n" + "="*60)
print("ETL Process Completed Successfully!")
print("="*60)

Database connection closed.

ETL Process Completed Successfully!


## Summary

### ETL Process Overview:

**Extract:**
- Loaded retail_store_sales.csv dataset

**Transform:**
- Converted empty strings to NaN for proper handling
- Converted numeric columns (Price Per Unit, Quantity, Total Spent) to appropriate data types
- Converted Transaction Date to datetime format
- Filled missing Price Per Unit with median grouped by Category
- Filled missing Quantity with median
- Recalculated Total Spent where missing
- Filled missing Item values with 'Unknown'
- Converted Discount Applied to boolean and filled missing with False
- Removed duplicate records

**Load:**
- Created SQLite3 database: retail_sales.db
- Loaded cleaned data into sales_data table
- Verified data integrity and statistics